# Yellow Sea SST: Phase 1 exploration

This notebook visualizes the processed NOAA OISST v2.1 regional time series. It does not download data, fit advanced models, or contain stored numerical results. Run the download and preprocessing commands in the repository README first.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from statsmodels.graphics.tsaplots import plot_acf

RANDOM_SEED = 42
sns.set_theme(style="whitegrid")

project_root = Path.cwd()
if not (project_root / "src").exists():
    project_root = project_root.parent
data_file = Path(os.environ.get(
    "SST_DATA_FILE",
    project_root / "data" / "processed" / "yellow_sea_sst_daily.csv",
))
if not data_file.exists():
    raise FileNotFoundError(
        f"Processed data not found at {data_file}. Run preprocessing first."
    )

In [ ]:
sst = pd.read_csv(data_file, parse_dates=["date"]).set_index("date")
required = {"sst_celsius", "climatology_celsius", "anomaly_celsius"}
missing = required.difference(sst.columns)
if missing:
    raise ValueError(f"Processed file is missing columns: {sorted(missing)}")
sst.info()

## Regional mean SST time series

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
sst["sst_celsius"].plot(ax=ax, linewidth=0.7)
ax.set(title="Yellow Sea regional mean SST", ylabel="SST (°C)", xlabel="Date")
plt.show()

## SST anomaly time series

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
sst["anomaly_celsius"].plot(ax=ax, linewidth=0.7)
ax.axhline(0, color="black", linewidth=0.8)
ax.set(title="Yellow Sea SST anomaly", ylabel="Anomaly (°C)", xlabel="Date")
plt.show()

## Monthly climatology

In [ ]:
monthly_climatology = sst.groupby(sst.index.month)["sst_celsius"].mean()
fig, ax = plt.subplots(figsize=(8, 4))
monthly_climatology.plot.bar(ax=ax, color="steelblue")
ax.set(title="Monthly SST climatology", xlabel="Month", ylabel="SST (°C)")
plt.show()

## Anomaly distribution

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(sst["anomaly_celsius"].dropna(), bins=50, kde=True, ax=ax)
ax.set(title="Distribution of SST anomalies", xlabel="Anomaly (°C)")
plt.show()

## Autocorrelation function

In [ ]:
anomaly = sst["anomaly_celsius"].dropna()
if len(anomaly) < 3:
    raise ValueError("At least three anomaly observations are required for ACF.")
max_lags = min(365, len(anomaly) // 2 - 1)
fig, ax = plt.subplots(figsize=(10, 4))
plot_acf(anomaly, lags=max_lags, ax=ax, zero=False)
ax.set(title="SST anomaly autocorrelation", xlabel="Lag (days)")
plt.show()